In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp
from scipy import stats
from scipy.optimize import curve_fit

Import the data:

In [2]:
data = {
    "Romulus_N2": pd.read_csv("../Data/Romulus_n2.csv"),
    "Romulus_O2": pd.read_csv("../Data/Romulus_o2.csv"),
    "Remus_N2": pd.read_csv("../Data/Remus_n2.csv"),
    "Remus_O2": pd.read_csv("../Data/Remus_o2.csv")
}


The linearised Langmuir equation is $\frac{p}{q}$ = $\frac{1}{{q_s}k}$ + $\frac{1}{q_s}p$, and the linearised Freundlich equation is $ln$ $q$ = $ln$ $k$ + $\frac{1}{m}$ $ln$ $p$.


We must first remove the row containing zeros. Then we transform the data into y and x.

In [3]:
P_raw = {}
q_raw = {}

x_langmuir = {}
y_langmuir = {}

x_freundlich = {}
y_freundlich = {}

for name, df in data.items():
    df = df[df["P [Pa]"] > 0]
    P = df["P [Pa]"].to_numpy()
    q = df["q [mol kg-1]"].to_numpy()
    P_raw[name] = P
    q_raw[name] = q
    x_langmuir[name] = P
    y_langmuir[name] = P/q
    x_freundlich[name] = np.log(P)
    y_freundlich[name] = np.log(q)

In [4]:
def langmuir_equation(P,qs,K):
    return qs*K*P / (1+K*P)
def freundlich_equation(P,m,K):
    return K*P**(1/m)

Fit the pure-component data to the linearised versions of the Langmuir and Freundlich isotherms and provide the functional form along with the estimated parameters (with the correct units). 

In [5]:
rows = []
for name in x_langmuir:
    results_langmuir = stats.linregress(x_langmuir[name],y_langmuir[name])
    results_freundlich = stats.linregress(x_freundlich[name],y_freundlich[name])

    qs_langmuir = 1 / results_langmuir.slope
    K_langmuir = 1 / (qs_langmuir * results_langmuir.intercept)

    m_freundlich = 1 / results_freundlich.slope
    K_freundlich = np.exp(results_freundlich.intercept)
    
    rows.append({
        "data_set": name,
        "slope_langmuir": results_langmuir.slope,
        "intercept_langmuir": results_langmuir.intercept,
        "R2_langmuir": results_langmuir.rvalue**2,
        "qs_langmuir": qs_langmuir,
        "K_langmuir": K_langmuir,
        "slope_freundlich": results_freundlich.slope,
        "intercept_freundlich": results_freundlich.intercept,
        "R2_freundlich": results_freundlich.rvalue**2,
        "m_freundlich": m_freundlich,
        "K_freundlich": K_freundlich,
    })

results = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.4e}".format)
display(results)

,data_set,slope_langmuir,intercept_langmuir,R2_langmuir,qs_langmuir,K_langmuir,slope_freundlich,intercept_freundlich,R2_freundlich,m_freundlich,K_freundlich
0,Romulus_N2,4.1649e-01,8.0222e+04,9.9654e-01,2.4010e+00,5.1917e-06,7.5151e-01,-9.1556e+00,9.8836e-01,1.3307e+00,1.0563e-04
1,Romulus_O2,3.3078e-01,5.6646e+05,9.2975e-01,3.0232e+00,5.8394e-07,9.2793e-01,-1.2550e+01,9.9989e-01,1.0777e+00,3.5452e-06
2,Remus_N2,3.5929e-01,3.5576e+05,9.9879e-01,2.7833e+00,1.0099e-06,7.7541e-01,-1.0278e+01,9.9509e-01,1.2896e+00,3.4365e-05
3,Remus_O2,2.4199e-01,9.1636e+05,9.2127e-01,4.1325e+00,2.6407e-07,9.2688e-01,-1.2918e+01,9.9988e-01,1.0789e+00,2.4524e-06


In [6]:
results = results.set_index("data_set")
results = results.sort_index()
results


,slope_langmuir,intercept_langmuir,R2_langmuir,qs_langmuir,K_langmuir,slope_freundlich,intercept_freundlich,R2_freundlich,m_freundlich,K_freundlich
data_set,,,,,,,,,,
Remus_N2,3.5929e-01,3.5576e+05,9.9879e-01,2.7833e+00,1.0099e-06,7.7541e-01,-1.0278e+01,9.9509e-01,1.2896e+00,3.4365e-05
Remus_O2,2.4199e-01,9.1636e+05,9.2127e-01,4.1325e+00,2.6407e-07,9.2688e-01,-1.2918e+01,9.9988e-01,1.0789e+00,2.4524e-06
Romulus_N2,4.1649e-01,8.0222e+04,9.9654e-01,2.4010e+00,5.1917e-06,7.5151e-01,-9.1556e+00,9.8836e-01,1.3307e+00,1.0563e-04
Romulus_O2,3.3078e-01,5.6646e+05,9.2975e-01,3.0232e+00,5.8394e-07,9.2793e-01,-1.2550e+01,9.9989e-01,1.0777e+00,3.5452e-06


calculate the standard error propogation of the slope and intercept errors ont the parameter voalues attained. Langmuir, from $q_s = 1/s$ and $K = s/c$:

$$\sigma_{q_s} = \frac{\sigma_s}{s^2}$$

$$\sigma_K = K\sqrt{\left(\frac{\sigma_s}{s}\right)^2 + \left(\frac{\sigma_c}{c}\right)^2 - \frac{2\,\mathrm{cov}(s,c)}{s\,c}}$$

Freundlich, from $m = 1/s$ and $K = e^{c}$:

$$\sigma_m = \frac{\sigma_s}{s^2}$$

$$\sigma_K = K\,\sigma_c$$

where $s$ is the slope, $c$ is the intercept, and $\sigma = \sqrt{\mathrm{variance}}$.

Then append these standard error values to the table above.

In [7]:
se_rows = []
for name in x_langmuir:
    # --- Langmuir: fit with polyfit to get the covariance matrix ---
    (s, c), cov = np.polyfit(x_langmuir[name], y_langmuir[name], 1, cov=True)
    sigma_s, sigma_c = np.sqrt(cov[0, 0]), np.sqrt(cov[1, 1])
    cov_sc = cov[0, 1]

    qs = 1 / s
    K = s / c
    se_qs_L = sigma_s / s**2
    se_K_L = abs(K) * np.sqrt((sigma_s / s)**2 + (sigma_c / c)**2 - 2 * cov_sc / (s * c))

    # --- Freundlich ---
    (s_f, c_f), cov_f = np.polyfit(x_freundlich[name], y_freundlich[name], 1, cov=True)
    sigma_s_f, sigma_c_f = np.sqrt(cov_f[0, 0]), np.sqrt(cov_f[1, 1])

    K_f = np.exp(c_f)
    se_m_F = sigma_s_f / s_f**2
    se_K_F = K_f * sigma_c_f

    se_rows.append({
        "se_qs_langmuir": se_qs_L,
        "se_K_langmuir": se_K_L,
        "se_m_freundlich": se_m_F,
        "se_K_freundlich": se_K_F,
    })

se_df = pd.DataFrame(se_rows)
results_with_se = pd.concat([results, se_df], axis=1)
display(results_with_se)

,slope_langmuir,intercept_langmuir,R2_langmuir,qs_langmuir,K_langmuir,slope_freundlich,intercept_freundlich,R2_freundlich,m_freundlich,K_freundlich,se_qs_langmuir,se_K_langmuir,se_m_freundlich,se_K_freundlich
Remus_N2,3.5929e-01,3.5576e+05,9.9879e-01,2.7833e+00,1.0099e-06,7.7541e-01,-1.0278e+01,9.9509e-01,1.2896e+00,3.4365e-05,NaN,NaN,NaN,NaN
Remus_O2,2.4199e-01,9.1636e+05,9.2127e-01,4.1325e+00,2.6407e-07,9.2688e-01,-1.2918e+01,9.9988e-01,1.0789e+00,2.4524e-06,NaN,NaN,NaN,NaN
Romulus_N2,4.1649e-01,8.0222e+04,9.9654e-01,2.4010e+00,5.1917e-06,7.5151e-01,-9.1556e+00,9.8836e-01,1.3307e+00,1.0563e-04,NaN,NaN,NaN,NaN
Romulus_O2,3.3078e-01,5.6646e+05,9.2975e-01,3.0232e+00,5.8394e-07,9.2793e-01,-1.2550e+01,9.9989e-01,1.0777e+00,3.5452e-06,NaN,NaN,NaN,NaN
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0824e-02,2.0969e-07,4.1691e-02,2.7842e-05
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.1551e-01,9.4612e-08,5.5834e-03,2.0260e-07
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.4315e-02,1.7458e-08,3.2014e-02,8.2611e-06
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.2711e-01,3.0262e-08,4.1042e-03,1.0765e-07


This is sufficient evidence to fit the data to isotherms.

Units: $P$ in Pa, $q$ in mol kg$^{-1}$. Parameters come from the linearised fits; $\pm$ values are one standard error.

### N$_2$: Langmuir model

$$q = \frac{q_s K P}{1 + K P}$$

**Romulus N$_2$:** $q_s = 2.40 \pm 0.04$ mol kg$^{-1}$, $K = (5.19 \pm 0.21)\times10^{-6}$ Pa$^{-1}$

$$q = \frac{1.25\times10^{-5}\,P}{1 + 5.19\times10^{-6}\,P}$$

**Remus N$_2$:** $q_s = 2.78 \pm 0.03$ mol kg$^{-1}$, $K = (1.010 \pm 0.017)\times10^{-6}$ Pa$^{-1}$

$$q = \frac{2.81\times10^{-6}\,P}{1 + 1.01\times10^{-6}\,P}$$

### O$_2$: Freundlich model

$$q = K P^{1/m}$$

**Romulus O$_2$:** $m = 1.078 \pm 0.006$, $K = (3.55 \pm 0.20)\times10^{-6}$ mol kg$^{-1}$ Pa$^{-1/m}$

$$q = 3.55\times10^{-6}\,P^{0.928}$$

**Remus O$_2$:** $m = 1.079 \pm 0.004$, $K = (2.45 \pm 0.11)\times10^{-6}$ mol kg$^{-1}$ Pa$^{-1/m}$

$$q = 2.45\times10^{-6}\,P^{0.927}$$

### Predicted uptake at 1 bar

| | N$_2$ (Langmuir) | O$_2$ (Freundlich) | N$_2$/O$_2$ |
|---|---|---|---|
| Romulus | 0.82 mol/kg | 0.155 mol/kg | 5.3 |
| Remus | 0.255 mol/kg | 0.106 mol/kg | 2.4 |

*Valid only within the measured pressure range.*

**Component mass balance**
$$\frac{\partial y_i}{\partial t} = D_L \frac{\partial^2 y_i}{\partial z^2} - \frac{\partial (y_i v)}{\partial z} - \frac{R T \rho_s (1-\varepsilon)}{P\,\varepsilon}\frac{\partial q_i}{\partial t}, \qquad \sum_i y_i = 1 \tag{1}$$

**Axial dispersion**
$$D_L = 0.7 D_m + v_{in} r_p \tag{2}$$

**Boundary conditions**
$$z = 0: \quad y_i(z=0) = y_{i,feed} + \frac{D_L}{v}\left.\frac{\partial y_i}{\partial z}\right|_{z=0} \tag{3}$$

$$z = L: \quad \left.\frac{\partial y_i}{\partial z}\right|_{z=L} = 0 \tag{4}$$

**Solid phase mass balance (LDF)**
$$\frac{\partial q_i}{\partial t} = k_i\left(q_i^{*} - q_i\right) \tag{5}$$

In [13]:
qs_Romulus_N2 = results.loc["Romulus_N2", "qs_langmuir"]
K_Romulus_N2 = results.loc["Romulus_N2", "K_langmuir"]
m_Romulus_N2 = results.loc["Romulus_N2", "m_freundlich"]
K_Romulus_N2 = results.loc["Romulus_N2", "K_freundlich"]
qs_Remus_N2 = results.loc["Remus_N2", "qs_langmuir"]
K_Remus_N2 = results.loc["Remus_N2", "K_langmuir"]
m_Remus_N2 = results.loc["Remus_N2", "m_freundlich"]
K_Remus_N2 = results.loc["Remus_N2", "K_freundlich"]
print(qs_Romulus_N2, K_Romulus_N2, m_Romulus_N2, K_Romulus_N2)
print(qs_Remus_N2, K_Remus_N2, m_Remus_N2, K_Remus_N2)

2.4010218464535713 0.00010562505150316068 1.330650329909683 0.00010562505150316068
2.783262780230972 3.436542635456928e-05 1.2896403920083583 3.436542635456928e-05


In [8]:
# Column dimensions
column_length = 0.32 # [m] L
L = column_length
column_void_fraction = 0.32 # [dimensionless] epsilon
eps = column_void_fraction
Particle_density_of_adsorbents_A_and_B = 990 # [kg/m^3] rho_s
rho_s = Particle_density_of_adsorbents_A_and_B
Particle_radius = 0.001 # [m] r_p
r_p = Particle_radius
# Properties and constants
Tortuosity = 3 # [dimensionless] tau
tau = Tortuosity
Molecular_diffussion_D_m = 7.481e-5 # [m^2/s] D_m
D_m = Molecular_diffussion_D_m
Universal_gas_constant_R = 8.314 # [m^3 Pa/mol/K] R
R = Universal_gas_constant_R
# Operating Conditions
Process_temperature_T = 303 # [K] T
T = Process_temperature_T
Process_pressure_P = 100000 # [Pa] P
P = Process_pressure_P      
inlet_superficial_velocity_u_in = 4.864e-3 # [m/s] u_in
u_in = inlet_superficial_velocity_u_in

In [9]:
# Calculate Particle Diffusivity
D_p = D_m / tau
# Assumption to use plain Glueckauf model
k = 15 * D_p / (r_p**2)
# Calculate inlet interstitial velocity
v_in = u_in / eps
# Calculate Dispersion coefficient
D_L = 0.7 * D_m + v_in * r_p

In [10]:
# Define the Grid
N = 600 # [dimensionless] number of nodes
z = np.linspace(0, L, N)

In [ ]:
def ldf_model(
    t, 
    qi,
    qi_star: float):

    dqi_dt = k * (qi_star - qi)

    return dqi_dt

def gas_speed(dqi_dt):

    # solve the sum of the dq_i/dt from the ldf_model

    dv_dz = (- R * T * rho_s * (1 - eps) / (P * eps)) * dqi_dt
    return dv_dz

def column_model(
    t, 
    state,
    lengthArray: np.ndarray,
    ):

    # extract the concentration and adsorption distribution from the state vector
    y_N2 = state[0:N]
    y_O2 = state[N:2*N]
    q_N2 = state[2*N:3*N]
    q_O2 = state[3*N:4*N]

    # add ghost cells (boundary conditions) to current state
    concentration_distribution = np.insert(y, 0)

    # determine number of points in the pipe
    N_points = lengthArray.size
    # initialize an array to store derivatives
    dyi_dt = np.zeros(N_points)

    p_N2 = P * y_N2
    p_O2 = P * y_O2
    qi_star_N2 = langmuir_equation(p_N2, qs=q_N2, K=K_N2)
    qi_star_O2 = freundlich_equation(p_O2, m=m_O2, K=K_O2)

    dqi_dt = ldf_model(t, qi, qi_star_N2)
    dv_dz = gas_speed(dqi_dt)
    return dyi_dt

sol = solve_ivp(column_model)


TypeError: solve_ivp() missing 2 required positional arguments: 't_span' and 'y0'